# Orchestration with Apache Airflow (and Its Alternatives)

Notebook for **Chapter 6** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch06.html

Tip: change a number, predict the output, then run the cell again.

## 6.1 The Ideas: DAGs, Schedules, Data Intervals, Retries

In [ ]:
import graphlib, datetime
fails_left = {"extract": 1}          # extract fails once, then works
state, outputs = {}, {}              # state: (task, date) -> 'success'
def extract(ds):
    if fails_left.get("extract", 0) > 0:
        fails_left["extract"] -= 1; raise ConnectionError("source timeout")
    outputs[("raw", ds)] = [1, 2, 3]
def transform(ds): outputs[("clean", ds)] = [x * 10 for x in outputs[("raw", ds)]]
def load(ds):      outputs[("mart", ds)] = sum(outputs[("clean", ds)])     # overwrite by date key
TASKS = {"extract": (extract, []), "transform": (transform, ["extract"]), "load": (load, ["transform"])}
def run_dag(ds, retries=2):
    order = graphlib.TopologicalSorter({k: v[1] for k, v in TASKS.items()}).static_order()
    for t in order:
        if state.get((t, ds)) == "success": print(f"  {ds} {t}: skipped (already success)"); continue
        for attempt in range(1, retries + 2):
            try:
                TASKS[t][0](ds); state[(t, ds)] = "success"; print(f"  {ds} {t}: success (attempt {attempt})"); break
            except Exception as e:
                print(f"  {ds} {t}: attempt {attempt} failed: {e}")
        else:
            print(f"  {ds} {t}: FAILED, downstream not run"); return
start = datetime.date(2024, 6, 1)
for d in range(3):
    ds = (start + datetime.timedelta(days=d)).isoformat(); print("run for", ds); run_dag(ds)
print("clear one task and re-run (backfill/rerun):")
del state[("load", "2024-06-02")]; run_dag("2024-06-02")
print({k: v for k, v in outputs.items() if k[0] == "mart"})

## 6.2 Airflow in Practice: Code, Components and Best Practices

In [ ]:
# Airflow 2.x style (not run here). Check your version's import paths.
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator

default_args = {"owner": "data-eng", "retries": 2, "retry_delay": timedelta(minutes=5)}

def load_orders(ds, **_):
    # ds is the logical date as 'YYYY-MM-DD'. Overwrite that day's partition: idempotent.
    print(f"loading orders for {ds}")

with DAG(
    dag_id="daily_orders",
    start_date=datetime(2024, 1, 1),
    schedule="@daily",          # older versions: schedule_interval
    catchup=False,
    max_active_runs=1,
    default_args=default_args,
    tags=["shopkart"],
) as dag:
    extract  = BashOperator(task_id="extract", bash_command="python extract.py --date {{ ds }}")
    validate = PythonOperator(task_id="validate", python_callable=lambda **_: print("checks"))
    load     = PythonOperator(task_id="load", python_callable=load_orders)
    extract >> validate >> load           # >> means "then"

In [ ]:
# Airflow 2.x/3.x style (not run here). In Airflow 3 the imports come from airflow.sdk;
# in 2.x from airflow.decorators. Check the version you use.
from datetime import datetime
from airflow.decorators import dag, task

@dag(start_date=datetime(2024, 1, 1), schedule="@daily", catchup=False)
def orders_taskflow():
    @task
    def extract(ds=None):
        return {"day": ds, "rows": 1000}            # small metadata only!

    @task
    def validate(meta: dict):
        assert meta["rows"] > 0, "empty extract"
        return meta

    @task
    def load(meta: dict):
        print("loaded", meta["rows"], "rows for", meta["day"])

    load(validate(extract()))

orders_taskflow()

In [ ]:
@task
def list_files(): return ["a.csv", "b.csv", "c.csv"]
@task
def process(path): print("processing", path)
process.expand(path=list_files())        # makes 3 task instances at run time

In [ ]:
# not run here: a DAG import test with pytest
from airflow.models import DagBag
def test_no_import_errors():
    bag = DagBag(dag_folder="dags/", include_examples=False)
    assert bag.import_errors == {}, bag.import_errors
def test_daily_orders_has_retries():
    dag = DagBag(dag_folder="dags/", include_examples=False).get_dag("daily_orders")
    assert dag.default_args["retries"] >= 1

## 6.3 Alternatives, Event-Driven Triggers and Lineage

In [ ]:
from dagster import asset
@asset
def raw_orders(): ...
@asset
def clean_orders(raw_orders): ...          # depends on raw_orders by argument name
@asset
def revenue_by_city(clean_orders): ...

## 6.4 Airflow in Production: Sensors, Mapping, Assets, Pools and Backfills

In [ ]:
files = [f"part-{i}.csv" for i in range(5)]
attempts = {f: 0 for f in files}

def load_one(path):
    attempts[path] += 1
    if path == "part-3.csv" and attempts[path] == 1:
        raise ConnectionError("source hiccup")          # fails on its first try only
    return 100                                           # rows loaded

results = {}
for map_index, f in enumerate(files):                    # fan out: one instance per file
    for try_no in range(1, 4):                           # retries=2 means up to 3 tries
        try:
            results[map_index] = load_one(f); break
        except ConnectionError:
            print(f"map_index {map_index} ({f}) failed on try {try_no}, retrying")
print("instances:", len(results), "attempts per file:", list(attempts.values()))
print("fan in, total rows:", sum(results.values()))

map_index 3 (part-3.csv) failed on try 1, retrying
instances: 5 attempts per file: [1, 1, 1, 2, 1]
fan in, total rows: 500


In [ ]:
from datetime import date, timedelta
import math

start, end = date(2024, 6, 1), date(2024, 6, 5)
runs = []
d = start
while d <= end:
    runs.append((d, d + timedelta(days=1)))               # (logical date, interval end)
    d += timedelta(days=1)
for logical, interval_end in runs:
    print(f"run logical_date={logical}  covers [{logical}, {interval_end})")
max_active_runs = 2
print("runs:", len(runs), " rounds with max_active_runs=2:", math.ceil(len(runs) / max_active_runs))

# the same backfill with two loaders; sales_by_day[day] holds that day's revenue
source = {date(2024, 6, 1) + timedelta(days=i): 1000 + 100 * i for i in range(5)}

def good_task(table, logical):                            # overwrite the partition for the logical date
    table[logical] = source[logical]

bad_calls = []
def bad_task(table, logical, today=date(2024, 6, 20)):    # uses "today" instead of the logical date
    table[today] = table.get(today, 0) + source[logical]
    bad_calls.append(logical)

good, bad = {}, {}
for logical, _ in runs:
    good_task(good, logical); bad_task(bad, logical)
good_task(good, date(2024, 6, 3))                         # rerun one day: no change
print("good table:", {str(k): v for k, v in good.items()})
print("bad table :", {str(k): v for k, v in bad.items()})
print("good total", sum(good.values()), "source total", sum(source.values()))

run logical_date=2024-06-01  covers [2024-06-01, 2024-06-02)
run logical_date=2024-06-02  covers [2024-06-02, 2024-06-03)
run logical_date=2024-06-03  covers [2024-06-03, 2024-06-04)
run logical_date=2024-06-04  covers [2024-06-04, 2024-06-05)
run logical_date=2024-06-05  covers [2024-06-05, 2024-06-06)
runs: 5  rounds with max_active_runs=2: 3
good table: {'2024-06-01': 1000, '2024-06-02': 1100, '2024-06-03': 1200, '2024-06-04': 1300, '2024-06-05': 1400}
bad table : {'2024-06-20': 6000}
good total 6000 source total 6000


## Chapter recap

## Practice set

**1.** (easy) What does DAG stand for? **Answer:** Directed Acyclic Graph.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch06.html